In [1]:
import os
import json
from PyPDF2 import PdfReader
import ollama

In [2]:
OUTPUT_FILE = "train_data.jsonl"

In [3]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    return "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])


In [4]:
def generate_example(study_text, prev_text):
    prompt = f"""
You are an expert in university-level exam question creation. 
Generate a new question paper that strictly follows the same structure and difficulty of the previous question paper.
Replace old questions with new ones from the study material. Keep formatting unchanged.

### Previous Exam Paper:
{prev_text}

### Study Material:
{study_text}

Generate the new paper below:
"""
    response = ollama.chat(model="mistral", messages=[{"role": "user", "content": prompt}])
    return response['message']['content'].strip()

In [5]:
def add_pair_to_dataset(study_pdf_path, prev_pdf_path):
    try:
        study_text = extract_text_from_pdf(study_pdf_path)
        prev_text = extract_text_from_pdf(prev_pdf_path)

        print(f"✅ Processing:\nStudy Material: {study_pdf_path}\nPrevious Paper: {prev_pdf_path}")
        generated_output = generate_example(study_text, prev_text)

        entry = {
            "prompt": f"Study Material:\n{study_text}\n\nPrevious Paper:\n{prev_text}",
            "response": generated_output
        }

        with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
            entry = {
                "prompt": f"Study Material:\n{study_text}\n\nPrevious Paper:\n{prev_text}",
                "response": generated_output
            }
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        print(f"\n✅ Example successfully added to {OUTPUT_FILE}\n")

    except Exception as e:
        print(f"❌ Error: {e}")

In [6]:
if __name__ == "__main__":
    study_pdf_path = input("Enter path to Study Material PDF: ").strip()
    prev_pdf_path = input("Enter path to Previous Question Paper PDF: ").strip()

    if not os.path.exists(study_pdf_path):
        print("❌ Study Material PDF not found.")
    elif not os.path.exists(prev_pdf_path):
        print("❌ Previous Question Paper PDF not found.")
    else:
        add_pair_to_dataset(study_pdf_path, prev_pdf_path)

Enter path to Study Material PDF:  /home/anjana/Project/question_generation/materials/c_study.pdf
Enter path to Previous Question Paper PDF:  /home/anjana/Project/question_generation/materials/c_prev.pdf


✅ Processing:
Study Material: /home/anjana/Project/question_generation/materials/c_study.pdf
Previous Paper: /home/anjana/Project/question_generation/materials/c_prev.pdf

✅ Example successfully added to train_data.jsonl

